# Раздел 1 · Введение в Qdrant и векторный поиск

In [1]:
%pip install sentence-transformers numpy ipykernel --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 20.6 MB/s eta 0:00:00


In [3]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("./books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}

print("книг в подборке:", len(BOOKS))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

книг в подборке: 87


## Эмбеддинги

У моделей семейства E5 (в том числе у нашей `multilingual-e5-small`) обязательные текстовые префиксы: `"query: "` для запросов, `"passage: "` для документов — без них качество тихо просядет.

### Пример

Два лёгких юмористических рассказа Чехова должны по эмбеддингу оказаться ближе друг к другу, чем любой из них — к эпическому и трагическому роману-эпопее "Война и мир".

In [7]:
def embed_passages(texts):
    # у модели E5 обязательный префикс "passage: " для ДОКУМЕНТОВ (не для запросов) - без него качество тихо просядет
    prefixed = ["passage: " + t for t in texts]
    return model.encode(prefixed, normalize_embeddings=True).tolist()


def embed_query(text):
    # а для ЗАПРОСОВ префикс другой - "query: "; перепутать их местами - частая ошибка с моделями семейства E5
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    # косинусная близость вручную, чтобы увидеть формулу за score, который потом отдаёт сам Qdrant
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


a, b, c = BOOK_BY_ID[42]["text"], BOOK_BY_ID[43]["text"], BOOK_BY_ID[15]["text"]
va, vb, vc = np.array(embed_passages([a, b, c]))

print("Хамелеон <-> Толстый и тонкий:", round(cos(va, vb), 3))
print("Хамелеон <-> Война и мир:     ", round(cos(va, vc), 3))


Хамелеон <-> Толстый и тонкий: 0.852
Хамелеон <-> Война и мир:      0.813


Реальный вывод: `Хамелеон <-> Толстый и тонкий: 0.852`, `Хамелеон <-> Война и мир: 0.813` — разница
не гигантская (все три текста всё же на русском про человеческие отношения), но направление верное.

## Задание 1

Та же идея на другой паре: два тяжёлых романа Достоевского — "Преступление и наказание" (`id=10`)
и "Идиот" (`id=11`) — должны быть ближе друг к другу, чем "Преступление и наказание" — к короткому
юмористическому "Хамелеону" (`id=42`). Посчитайте `sim_two_dostoevsky` и `sim_dostoevsky_khameleon`
(используйте уже готовые `embed_passages`/`cos` из примера).

**Используйте:** `embed_passages(...)`, `cos(...)` — это упражнение вообще не обращается к Qdrant,
только к самой модели эмбеддингов (`model.encode` под капотом).

In [11]:
# ВАШ КОД ЗДЕСЬ
dost_text_1 = BOOK_BY_ID[10]["text"]
dost_text_2 = BOOK_BY_ID[11]["text"]
hameleon_text = BOOK_BY_ID[42]["text"]

va, vb, vc = np.array(embed_passages([dost_text_1, dost_text_2, hameleon_text]))

sim_two_dostoevsky = round(cos(va, vb), 3)
sim_dostoevsky_khameleon = round(cos(va, vc), 3)

print("dost & dost:", sim_two_dostoevsky)
print("dost & hameleon:", sim_dostoevsky_khameleon)

dost & dost: 0.845
dost & hameleon: 0.833


In [12]:
def check_ex1(sim_two_dostoevsky, sim_dostoevsky_khameleon):
    # косинусная близость нормализованных эмбеддингов всегда в [-1, 1], а для двух текстов на
    # одну и ту же общую тему (русская проза) она реалистично лежит в [0.5, 1.0] - если у вас
    # число сильно вне этого диапазона, скорее всего где-то перепутаны id или забыт np.array(...)
    for name, value in [("sim_two_dostoevsky", sim_two_dostoevsky), ("sim_dostoevsky_khameleon", sim_dostoevsky_khameleon)]:
        assert 0.5 <= value <= 1.0, (
            "{} = {:.3f} выглядит неправдоподобно для двух текстов на одну тему - "
            "проверьте id книг и что вектор из embed_passages(...) обёрнут в np.array(...)".format(name, value)
        )
    # два тяжёлых романа одного автора должны быть семантически ближе друг к другу,
    # чем один из них - к короткому юмористическому рассказу другого автора
    assert sim_two_dostoevsky > sim_dostoevsky_khameleon, (
        "два романа Достоевского должны быть ближе друг к другу ({:.3f}), чем к лёгкому "
        "юмористическому рассказу ({:.3f})".format(sim_two_dostoevsky, sim_dostoevsky_khameleon)
    )
    print("OK: {:.3f} > {:.3f}".format(sim_two_dostoevsky, sim_dostoevsky_khameleon))
    print("ОТВЕТ ДЛЯ STEPIK:", round(sim_two_dostoevsky, 3))


check_ex1(sim_two_dostoevsky, sim_dostoevsky_khameleon)


OK: 0.845 > 0.833
ОТВЕТ ДЛЯ STEPIK: 0.845
